In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from utils import load_metrics_df

graph_name = "amazon"
split = "test"

embedding_model = "azure/text-embedding-3-large"
embedding_model_short = embedding_model.split("/")[-1]

search_modes = ["bm25", "embeddings"]
metric_keys = ["hit@1", "hit@5", "recall@20", "MRR"]
metric_labels = ["Hit@1", "Hit@5", "Recall@20", "MRR"]
sorting = "voting"

for base_model_name in [
    "graph_explorer_gpt-4.1",
    # "graph_explorer_gpt-5-mini",
    "graph_explorer_gpt-5.2",
]:
    mode_dirs = {}
    for mode in search_modes:
        suffix = f"_embeddings_{embedding_model_short}" if mode == "embeddings" else ""
        mode_dirs[mode] = Path(
            f"../data/experiments/{graph_name}/{base_model_name}{suffix}/{split}"
        )

    mode_dfs = {}
    for mode in search_modes:
        if not mode_dirs[mode].exists():
            print(f"  [{mode}] directory not found: {mode_dirs[mode]}")
            continue
        mode_dfs[mode] = load_metrics_df(mode_dirs[mode], max_agents=3, sorting_criteria=sorting)

    if len(mode_dfs) == 0:
        print(f"  Skipping {base_model_name}: no results found\n")
        continue

    available_modes = list(mode_dfs.keys())
    print(f"Model: {base_model_name}")

    if len(available_modes) == 2:
        common_ids = set(mode_dfs["bm25"]["question_id"]) & set(mode_dfs["embeddings"]["question_id"])
        print(
            f"  Common questions: {len(common_ids)}  (bm25={len(mode_dfs['bm25'])}, embeddings={len(mode_dfs['embeddings'])})"
        )
        filtered = {
            mode: df[df["question_id"].isin(common_ids)].reset_index(drop=True)
            for mode, df in mode_dfs.items()
        }
    else:
        filtered = mode_dfs
        print(f"  Questions: {len(list(filtered.values())[0])}  (modes: {', '.join(available_modes)})")

    for label, key in zip(metric_labels, metric_keys):
        vals = {mode: float(round(filtered[mode][key].mean(), 4)) for mode in available_modes}
        parts = "  ".join(f"{mode}={vals[mode]}" for mode in available_modes)
        print(f"  {label}: {parts}")
    print()

    # fig, axes = plt.subplots(1, len(metric_keys), figsize=(5 * len(metric_keys), 4))
    # for ax, (label, key) in zip(axes, zip(metric_labels, metric_keys)):
    #     for mode in search_modes:
    #         df_mode = filtered[mode]
    #         running_mean = [df_mode[key].iloc[:i].mean() for i in range(1, len(df_mode) + 1)]
    #         ax.plot(running_mean, label=mode)
    #         ax.axhline(y=df_mode[key].mean(), linestyle="--", alpha=0.5)
    #     ax.set_title(label)
    #     ax.set_xlabel("# questions")
    #     ax.legend()
    # plt.suptitle(f"{base_model_name} — BM25 vs Embeddings (n={len(common_ids)})")
    # plt.tight_layout()
    # plt.show()

Model: graph_explorer_gpt-4.1
  Common questions: 1640  (bm25=1641, embeddings=1641)
  Hit@1: bm25=0.564  embeddings=0.4713
  Hit@5: bm25=0.7622  embeddings=0.6994
  Recall@20: bm25=0.6058  embeddings=0.5813
  MRR: bm25=0.6544  embeddings=0.5745

  [embeddings] directory not found: ../data/experiments/amazon/graph_explorer_gpt-5.2_embeddings_text-embedding-3-large/test
Model: graph_explorer_gpt-5.2
  Questions: 1641  (modes: bm25)
  Hit@1: bm25=0.5612
  Hit@5: bm25=0.7776
  Recall@20: bm25=0.5862
  MRR: bm25=0.6571



In [2]:
plt.plot([df["hit@1"].iloc[:i].mean() for i in range(1, len(df) + 1)], label="Hit@1")
plt.axhline(y=df["hit@1"].mean(), color="r", linestyle="--", label="Hit@1 Mean")
plt.plot([df["recall@20"].iloc[:i].mean() for i in range(1, len(df) + 1)], label="Recall@20")
plt.axhline(y=df["recall@20"].mean(), color="g", linestyle="--", label="Recall@20 Mean")
plt.legend()

NameError: name 'df' is not defined

In [ ]:
df['combined_answer_indices'].apply(len).hist()